# parsing.ms_office.markitdown.win

> Windows-specific image post-processing around the shared MarkItDown Office workflow.

In [ ]:
# |default_exp parsing.ms_office.markitdown.win

In [ ]:
# | hide
from nbdev.showdoc import *

## Shared API

The platform-neutral conversion and extraction workflow lives in `parsing.ms_office.markitdown.utils`.

In [ ]:
# | export
import os
import re
import shutil
import subprocess
from pathlib import Path

from ribosome.parsing.ms_office.markitdown import utils as _utils

ENV_FILE = _utils.ENV_FILE
HTML_DATA_IMAGE_RE = _utils.HTML_DATA_IMAGE_RE
IMAGE_EXTENSION_BY_MIME = _utils.IMAGE_EXTENSION_BY_MIME
MARKDOWN_DATA_IMAGE_RE = _utils.MARKDOWN_DATA_IMAGE_RE
OFFICE_EXTENSIONS = _utils.OFFICE_EXTENSIONS
PROJ_ROOT = _utils.PROJ_ROOT
convert_office_to_md = _utils.convert_office_to_md
extract_base64_from_md = _utils.extract_base64_from_md
extract_base64_images = _utils.extract_base64_images
extract_md_base64_images_win = _utils.extract_base64_images
get_office_files_root = _utils.get_office_files_root
process_office_files = _utils.process_office_files


## Windows ImageMagick helpers

In [ ]:
# | export
_GIF_IMAGE_RE = re.compile(
    r"(?P<prefix>!\[[^\]]*\]\()(?P<path>[^)\n]+?\.gif)(?P<suffix>\))",
    flags=re.IGNORECASE,
)


def _magick_convert(source: Path, target: Path) -> None:
    """Convert one image with the Windows ImageMagick executable."""
    subprocess.run(
        [
            "magick",
            "-units",
            "PixelsPerInch",
            str(source),
            "-density",
            "300",
            "-trim",
            "-border",
            "5",
            str(target),
        ],
        check=True,
    )


def convert_md_gif2png_win(
    markdown_file_path: Path | str,
    image_output_folder: Path | str = ".",
) -> int:
    """Convert linked GIF images to PNG and rewrite their Markdown links."""
    markdown_file = Path(markdown_file_path).expanduser().resolve()
    if not markdown_file.is_file():
        print(f"Error: Markdown file not found at {markdown_file}")
        return -1

    content = markdown_file.read_text(encoding="utf-8")
    converted = 0

    def replace_gif(match: re.Match) -> str:
        nonlocal converted
        linked_path = Path(match.group("path"))
        gif_file = (
            linked_path
            if linked_path.is_absolute()
            else markdown_file.parent / linked_path
        )
        if not gif_file.is_file():
            fallback = markdown_file.parent / image_output_folder / linked_path.name
            if fallback.is_file():
                gif_file = fallback
        png_file = gif_file.with_suffix(".png")
        _magick_convert(gif_file, png_file)
        converted += 1
        relative_png = Path(os.path.relpath(png_file, markdown_file.parent)).as_posix()
        return f'{match.group("prefix")}{relative_png}{match.group("suffix")}'

    rewritten = _GIF_IMAGE_RE.sub(replace_gif, content)
    if converted:
        markdown_file.write_text(rewritten, encoding="utf-8")
    return converted if converted else -1


def convert_gif2png_from_md(root_folder: Path | str) -> dict[str, list]:
    """Convert linked GIF images in every Markdown file below root."""
    root = Path(root_folder).expanduser().resolve()
    report = {"converted": [], "skipped": [], "failed": []}
    for markdown_file in sorted(path for path in root.rglob("*.md") if path.is_file()):
        try:
            count = convert_md_gif2png_win(markdown_file, "img")
        except (OSError, subprocess.CalledProcessError) as error:
            report["failed"].append((markdown_file, error))
            print(f"Failed to convert {markdown_file}: {error}")
            continue
        if count < 0:
            report["skipped"].append(markdown_file)
        else:
            report["converted"].append((markdown_file, count))
    return report


In [ ]:
# | export
_HTML_VECTOR_IMAGE_RE = re.compile(
    r"(?P<prefix><img\b[^>]*?\bsrc\s*=\s*(?P<quote>[\"']))"
    r"(?P<path>[^\"']+?\.(?P<suffix>wmf|emf))"
    r"(?P<tail>(?P=quote)[^>]*>)",
    flags=re.IGNORECASE,
)


def extract_md_html_images_win(markdown_file_path: Path | str) -> int:
    """Convert linked WMF/EMF HTML images to PNG and make links relative."""
    markdown_file = Path(markdown_file_path).expanduser().resolve()
    if not markdown_file.is_file():
        return -1

    content = markdown_file.read_text(encoding="utf-8")
    converted = 0

    def replace_vector(match: re.Match) -> str:
        nonlocal converted
        linked_path = Path(match.group("path"))
        source = (
            linked_path
            if linked_path.is_absolute()
            else markdown_file.parent / linked_path
        )
        svg_file = source.with_suffix(".svg")
        png_file = source.with_suffix(".png")
        _magick_convert(source, svg_file)
        _magick_convert(source, png_file)
        converted += 1
        relative_png = Path(os.path.relpath(png_file, markdown_file.parent)).as_posix()
        return f'{match.group("prefix")}{relative_png}{match.group("tail")}'

    rewritten = _HTML_VECTOR_IMAGE_RE.sub(replace_vector, content)
    if converted:
        markdown_file.write_text(rewritten, encoding="utf-8")
    return converted if converted else -1


def convert_html_wmf_emf_image_from_md(root_folder: Path | str) -> dict[str, list]:
    """Convert linked WMF/EMF images in every Markdown file below root."""
    root = Path(root_folder).expanduser().resolve()
    report = {"converted": [], "skipped": [], "failed": []}
    for markdown_file in sorted(path for path in root.rglob("*.md") if path.is_file()):
        try:
            count = extract_md_html_images_win(markdown_file)
        except (OSError, subprocess.CalledProcessError) as error:
            report["failed"].append((markdown_file, error))
            print(f"Failed to convert {markdown_file}: {error}")
            continue
        if count < 0:
            report["skipped"].append(markdown_file)
        else:
            report["converted"].append((markdown_file, count))
    return report


## Windows output-tree helper

In [ ]:
# | export
def copy_md_files(
    src_md_root: Path,
    dst_md_root: Path,
    bOverwrite: bool = True,
) -> dict[str, list]:
    """Copy per-document Markdown folders into another mirrored tree."""
    src_root = src_md_root.expanduser().resolve()
    dst_root = dst_md_root.expanduser().resolve()
    dst_root.mkdir(parents=True, exist_ok=True)
    report = {"copied": [], "skipped": [], "failed": []}

    for markdown_file in sorted(
        path for path in src_root.rglob("*.md") if path.is_file()
    ):
        src_folder = markdown_file.parent
        dst_folder = dst_root / src_folder.relative_to(src_root)
        try:
            if dst_folder.exists():
                if not bOverwrite:
                    report["skipped"].append(dst_folder)
                    continue
                shutil.rmtree(dst_folder)
            shutil.copytree(src_folder, dst_folder)
        except OSError as error:
            report["failed"].append((src_folder, error))
            continue
        report["copied"].append(dst_folder)
    return report


## Run on Windows

In [ ]:
# Run the complete workflow using OFFICE_FILES_ROOT from PROJ_ROOT/.env.
PROCESS_REPORT = process_office_files(overwrite=False, image_output_folder="img")
{
    "office_files_found": len(PROCESS_REPORT["conversion"]["discovered"]),
    "markdown_converted": len(PROCESS_REPORT["conversion"]["converted"]),
    "markdown_skipped": len(PROCESS_REPORT["conversion"]["skipped"]),
    "conversion_failures": len(PROCESS_REPORT["conversion"]["failed"]),
    "images_extracted": PROCESS_REPORT["extraction"]["images_extracted"],
    "extraction_failures": len(PROCESS_REPORT["extraction"]["failed"]),
    "output_root": str(PROCESS_REPORT["conversion"]["output_root"]),
}

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()